# Input

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import blosum as bl

from collections import defaultdict
from tqdm import tqdm
from omegaconf import OmegaConf

In [ ]:
sns.set_context('talk')

In [ ]:
config_filepath = 'my/path/to/config.yaml'
cfg = OmegaConf.load(config_filepath)

In [ ]:
# Define useful variables

cell_lines = cfg['cell_lines']
editors = cfg['editors']
target_genes = cfg['target_genes']
gene_effect_essential_threshold = cfg['gene_effect_essential_threshold']
fdr = cfg['fdr']
control_type_order = cfg['control_type_order']
control_order = cfg['control_order']
gene_name_correction = cfg['gene_name_correction']
DEPMAP2CELL_LINE = cfg['depmap_2_cell_line']
processed_files_dirpath = cfg['processed_files_dirpath']
figures_dirpath = cfg['figures_dirpath']

validated_ctrls_filepath = cfg['validated_ctrls_filepath']
vep_data_dirpath = cfg['vep_data_dirpath']
variant_prediction_dirpath = cfg['variant_prediction_dirpath']

In [ ]:
# Create folders
for dirpath in [processed_files_dirpath, figures_dirpath]:
    if not os.path.exists(dirpath):
        os.mkdir(dirpath)

# Load data

In [ ]:
# Load LFC data

lfc_dfs = []
for cell_line in cell_lines:
    for editor in editors:
        filepath = cfg['lfc_data_filepaths'][cell_line][editor]
        df = pd.read_csv(filepath, sep='\t')
        df['cell_line'] = cell_line
        df['editor'] = editor
        lfc_dfs.append(df)

data_df = pd.concat(lfc_dfs, ignore_index=True)

In [ ]:
# Load gene metadata

gene_dfs = []
for cell_line in cell_lines:
    for editor in editors:
        filepath = cfg['gene_data_filepaths'][cell_line][editor]
        df = pd.read_csv(filepath, sep='\t')
        df['cell_line'] = cell_line
        df['editor'] = editor
        gene_dfs.append(df)

gene_data_df = pd.concat(gene_dfs, ignore_index=True)

In [ ]:
# Load control data

filepath = cfg['controls_filepath']
ctrls_df = pd.read_csv(filepath, index_col=0)

In [ ]:
# Load mutations (single and multiple nucleotide) computed with VEP
# and variant predictions computed with Cagiada model

single_vep_dfs = []
multiple_vep_dfs = []
multiple_match_dfs = []
variant_prediction_dfs = []

str_columns = ['STRAND', 
               'GENE_PHENO', 
               'TSL',
               'gnomADe_AF', 
               'gnomADe_AFR_AF', 
               'gnomADe_AMR_AF', 
               'gnomADe_ASJ_AF', 
               'gnomADe_EAS_AF', 
               'gnomADe_FIN_AF', 
               'gnomADe_MID_AF', 
               'gnomADe_NFE_AF', 
               'gnomADe_REMAINING_AF', 
               'gnomADe_SAS_AF', 
               'MAX_AF',
               'cDNA_position']

gene_name_correction = cfg['gene_name_correction']
new_to_old_gene_names = {v: k for k, v in gene_name_correction.items()}

for gene in tqdm(target_genes):
    for editor in editors:

        # Single edit
        filename = f'single_edits_{gene}_{editor}_out.csv'
        filepath = os.path.join(vep_data_dirpath, filename)
        df = pd.read_csv(filepath, sep='\t', dtype={col: str for col in str_columns})
        df['editor'] = editor
        single_vep_dfs.append(df)
        
        breakpoint()

        # Multiple edit
        filename = f'multiple_edits_{gene}_{editor}_out.csv'
        filepath = os.path.join(vep_data_dirpath, filename)
        df = pd.read_csv(filepath, sep='\t', dtype={col: str for col in str_columns})
        df['editor'] = editor
        multiple_vep_dfs.append(df)

        # Multiple edit matchs
        filename = f'multiple_edits_{gene}_{editor}_match.csv'
        filepath = os.path.join(vep_data_dirpath, filename)
        df = pd.read_csv(filepath, dtype={col: str for col in str_columns})
        df['editor'] = editor
        multiple_match_dfs.append(df)

    # Apply gene name correction to find correct file in computed variant prediction
    if gene in new_to_old_gene_names:
        if gene != 'PRR5L':
            gene_for_path = new_to_old_gene_names[gene]
        else:
            gene_for_path = gene
    else:
        gene_for_path = gene

    # Variant prediction
    filename = f'test_variant_predictions.csv'
    filepath = os.path.join(variant_prediction_dirpath, gene_for_path, filename)
    df = pd.read_csv(filepath)
    df['Gene'] = gene
    variant_prediction_dfs.append(df)

In [ ]:
single_vep_df = pd.concat(single_vep_dfs, ignore_index=True)
multiple_vep_df = pd.concat(multiple_vep_dfs, ignore_index=True)
multiple_match_df = pd.concat(multiple_match_dfs, ignore_index=True)
variant_pred_df = pd.concat(variant_prediction_dfs, ignore_index=True)

In [ ]:
columns_to_keep = [
    '#Uploaded_variation',
    'Location',
    'Gene',
    'Feature',
    'Feature_type',
    'Consequence',
    'cDNA_position',
    'CDS_position',
    'Protein_position',
    'Amino_acids',
    'Codons',
    'STRAND',
    'SYMBOL',
    'SIFT',
    'PolyPhen',
    'EXON',
    'INTRON',
    'editor',
]

single_vep_df = single_vep_df[columns_to_keep]
multiple_vep_df = multiple_vep_df[columns_to_keep]

In [ ]:
filepath = cfg['depmap_data_filepath']
depmap_gene_effects = pd.read_csv(filepath)

In [ ]:
# Load data on used guides (target genes and controls)

str_columns = ['guide_in_CDS']
filepath = cfg['guide_data_filepath']
all_guides_df = pd.read_csv(filepath, dtype={col: str for col in str_columns})

In [ ]:
# Get original BEstimate information
# to get all single Edit_Location (e.g. 104792640, 104792641, 104792642) 
# and Strand (-1 if antisense, 1 if sense)
# This also includes non-canonical guides, which we will then filter out using the guide list

edit_dfs = []
BESTIMATE_PATH = cfg['bestimate_data_dirpath']
for gene in target_genes:
    for editor in editors:
        edit_filepath = os.path.join(BESTIMATE_PATH, f'{gene}_{editor}_edit_df.csv')
        edit_df = pd.read_csv(edit_filepath)
        edit_df['editor'] = editor
        edit_dfs.append(edit_df)

edit_df = pd.concat(edit_dfs)

In [ ]:
columns_to_keep = [
    'CRISPR_PAM_Sequence',
    'Location',
    'Edit_Location',
    'Direction',
    'Strand',
    'Transcript_ID',
    'editor',
]

edit_df = edit_df[columns_to_keep]

In [ ]:
validated_ctrls = pd.read_excel(validated_ctrls_filepath)

# Link data

In [ ]:
# Link each guide to its corresponding edit information (e.g. Edit_Location, Strand) to get the corresponding mutations

print(all_guides_df.shape)
print(edit_df.shape)
guide_columns = ['gRNA_ID', 'CRISPR_PAM_Sequence', 'Transcript_ID']
guides_with_edit_df = all_guides_df[guide_columns].merge(edit_df, 
                                                         on=['CRISPR_PAM_Sequence', 'Transcript_ID'],
                                                         how='left')
guides_with_edit_df = guides_with_edit_df.rename({'gRNA_ID': 'sgRNA_ID', 'Hugo_Symbol': 'Gene'}, axis=1)
print(guides_with_edit_df.shape)

In [ ]:
# Check guides that have no A or C in the editing window

guide_with_no_AC_in_ew = guides_with_edit_df[guides_with_edit_df['editor'].isna()]
print(guide_with_no_AC_in_ew.shape)
guide_with_no_AC_in_ew.head()

In [ ]:
# Remove guides that have no A or C in the editing window
guides_with_edit_df = guides_with_edit_df.dropna(subset=['editor'])

In [ ]:
variant_pred_df['Original_AA'] = variant_pred_df['Mutation'].str[0]
variant_pred_df['Modified_AA'] = variant_pred_df['Mutation'].str[-1]
variant_pred_df['uniprot_rno'] = variant_pred_df['Mutation'].str[1:-1]

In [ ]:
# Some mutations were tested in duplicates, this could be corrected at the VEP step
# Potential TODO: remove vep duplication
print(multiple_vep_df.shape)
multiple_vep_df = multiple_vep_df.drop_duplicates().copy()
print(multiple_vep_df.shape)

In [ ]:
# Remove non-transcript VEP annotations (e.g. regulatory region, intergenic)
print(multiple_vep_df.shape)
multiple_vep_df = multiple_vep_df[multiple_vep_df['Feature_type'] == 'Transcript'].copy()
print(multiple_vep_df.shape)

In [ ]:
multiple_vep_df['chr'] = multiple_vep_df['#Uploaded_variation'].apply(lambda x: int(x.split('_')[0]))
multiple_vep_df['new_start_location'] = multiple_vep_df['Location'].apply(lambda x: int(x.split('-')[0].split(':')[1]))
multiple_vep_df['new_end_location'] = multiple_vep_df['Location'].apply(lambda x: int(x.split('-')[1]))
multiple_vep_df['vep_replacement'] = multiple_vep_df['#Uploaded_variation'].apply(lambda x: x.split('_')[-1])

multiple_match_df['chr'] = multiple_match_df['chr'].astype(int)
multiple_match_df['new_start_location'] = multiple_match_df['new_start_location'].astype(int)
multiple_match_df['new_end_location'] = multiple_match_df['new_end_location'].astype(int)
multiple_match_df['vep_replacement'] = multiple_match_df['vep_replacement'].astype(str)

In [ ]:
print(multiple_match_df.shape)

In [ ]:
# Adds the information to which guide the mutation is applied in
print(multiple_vep_df.shape)
multiple_vep_df = pd.merge(multiple_vep_df, multiple_match_df, on=['chr', 'new_start_location', 
                                                        'new_end_location', 'vep_replacement', 'editor'])
print(multiple_vep_df.shape)

In [ ]:
OFF_TARGETS_PATH = cfg['off_targets_dirpath']

ot_dfs = []
for cell_line in cell_lines:
    for gene in target_genes:
        ot_filepath = os.path.join(OFF_TARGETS_PATH, gene, f'{gene}_{cell_line}.csv')
        ot_df = pd.read_csv(ot_filepath)
        ot_df['cell_line'] = cell_line
        ot_dfs.append(ot_df)

ot_df = pd.concat(ot_dfs)
ot_df = ot_df.reset_index(drop=True)

In [ ]:
for cell_line in cell_lines:
    df = ot_df[(ot_df['cell_line'] == cell_line)]
    print(cell_line, editor)
    print('Fraction of off-target guides:', df['is_ot'].mean().tolist())
    print('Number of off-target guides:', df['is_ot'].sum().tolist())

In [ ]:
ot_df['has_essential'].value_counts()

In [ ]:
# Manual corrections for gene name
gene_data_df['id'] = gene_data_df['id'].replace(gene_name_correction)
for old_gene_name, new_gene_name in gene_name_correction.items():
    data_df['sgrna'] = data_df['sgrna'].str.replace(old_gene_name, new_gene_name)
    data_df['Gene'] = data_df['Gene'].str.replace(old_gene_name, new_gene_name)
    ot_df['sgRNA_ID'] = ot_df['sgRNA_ID'].str.replace(old_gene_name, new_gene_name)
    guides_with_edit_df['sgRNA_ID'] = guides_with_edit_df['sgRNA_ID'].str.replace(old_gene_name, new_gene_name)
    single_vep_df['SYMBOL'] = single_vep_df['SYMBOL'].str.replace(old_gene_name, new_gene_name)
    multiple_vep_df['SYMBOL'] = multiple_vep_df['SYMBOL'].str.replace(old_gene_name, new_gene_name)

In [ ]:
guides_with_edit_df

In [ ]:
# Adds Gene column
ot_df = ot_df[ot_df['sgRNA_ID'].isin(guides_with_edit_df['sgRNA_ID'])].copy()

In [ ]:
filepath = os.path.join(processed_files_dirpath, 'all_off_target_data.csv')
ot_df.to_csv(filepath, index=False)

In [ ]:
filepath = os.path.join(processed_files_dirpath, 'off_target_data.csv')
ot_df[ot_df['is_ot'].astype(bool)].to_csv(filepath, index=False)

In [ ]:
multiple_vep_df['Original_Sequence'] = multiple_vep_df['#Uploaded_variation'].apply(lambda x: x.split('_')[-1].split('/')[0])
multiple_vep_df['Modified_Sequence'] = multiple_vep_df['#Uploaded_variation'].apply(lambda x: x.split('_')[-1].split('/')[1]) 

In [ ]:
# Change Edit_Location to the single location that can be found in Location
# For further merge with guide information
single_vep_df['Edit_Location'] = single_vep_df['Location'].apply(lambda x: int(x.split(':')[1]))
single_vep_df = single_vep_df.drop('Location', axis=1)

In [ ]:
# Have a separate column for the Locations slice for multiple mutations
multiple_vep_df = multiple_vep_df.rename({'Location': 'Multiple_Location'}, axis=1)

In [ ]:
renamed_columns = {'Gene': 'ENSG_id'}
single_vep_df = single_vep_df.rename(renamed_columns, axis=1)
multiple_vep_df = multiple_vep_df.rename(renamed_columns, axis=1)

In [ ]:
# Important: a single Edit_Location can be found in multiple guides
# And same for the Multiple_Location for multiple mutations
print(single_vep_df.shape)
single_vep_df = single_vep_df.merge(guides_with_edit_df,
                                   left_on=['Edit_Location', 'Feature', 'editor'],
                                   right_on=['Edit_Location', 'Transcript_ID', 'editor'])
print(single_vep_df.shape)

In [ ]:
# We remove duplicate Edit_Location for the multiple mutations;
# As the edit location will be handled using the actual modified locations
print(guides_with_edit_df.shape)
unique_guides_editor = guides_with_edit_df.drop('Edit_Location', axis=1)
unique_guides_editor = unique_guides_editor.drop_duplicates(['CRISPR_PAM_Sequence', 'Transcript_ID', 'editor'])
print(unique_guides_editor.shape)

In [ ]:
print(multiple_vep_df.shape)
multiple_vep_df = multiple_vep_df.merge(unique_guides_editor,
                                       left_on=['CRISPR_PAM_Sequence', 'Feature', 'editor', 'sgRNA_ID'],
                                       right_on=['CRISPR_PAM_Sequence', 'Transcript_ID', 'editor', 'sgRNA_ID'])
print(multiple_vep_df.shape)

In [ ]:
def get_edit_locations(row):
    edit_locations = []
    start = row['new_start_location']
    end = row['new_end_location']
    locations = list(range(start, end + 1))
    original_seq = row['Original_Sequence']
    modified_seq = row['Modified_Sequence']
    direction = row['Direction']
    if direction == 'left':
        locations = list(reversed(locations))

    for location, original_n, modified_n in zip(locations, original_seq, modified_seq):
        if original_n != modified_n:
           edit_locations.append(location)

    return edit_locations

In [ ]:
multiple_vep_df['Edit_Location'] = multiple_vep_df.apply(get_edit_locations, axis=1)

# This will create one line per nucleotide location possibly mutated
print(multiple_vep_df.shape)
multiple_vep_df = multiple_vep_df.explode('Edit_Location')
print(multiple_vep_df.shape)

# Control data

In [ ]:
ctrls_df['sgRNA_type'].value_counts()

In [ ]:
positive_validated_ctrls = validated_ctrls[validated_ctrls['sgRNA_type'] == 'splice_essential']
g_guides = positive_validated_ctrls['G_guide'].unique()

validated_ctrls_df = ctrls_df[ctrls_df['G_sequence'].isin(g_guides) | (ctrls_df['sgRNA_type'] != 'splice_essential')]

In [ ]:
validated_ctrls_df['sgRNA_type'].value_counts()

In [ ]:
# Remove data from guides that has no A or C in the editing window
# And controls that should not work
print(data_df.shape)
data_df = data_df.rename({'sgrna': 'sgRNA_ID'}, axis=1)
canonical_data_df = data_df.merge(unique_guides_editor[['sgRNA_ID', 'editor']])

control_data_df = data_df.merge(validated_ctrls_df, left_on=['sgRNA_ID'], right_on=['Unique_ID'])

In [ ]:
good_data_df = pd.concat([canonical_data_df, control_data_df], ignore_index=True)
print(good_data_df.shape)

In [ ]:
mean_df = good_data_df.groupby(['cell_line', 'editor'])['LFC'].mean().reset_index().rename({'LFC': 'mean_lfc'}, axis=1)
std_df = good_data_df.groupby(['cell_line', 'editor'])['LFC'].std().reset_index().rename({'LFC': 'std_lfc'}, axis=1)

good_data_df = good_data_df.merge(mean_df, on=['cell_line', 'editor'])
good_data_df = good_data_df.merge(std_df, on=['cell_line', 'editor'])

good_data_df['zscore'] = (good_data_df['LFC'] - good_data_df['mean_lfc']) / good_data_df['std_lfc']

In [ ]:
target_gene_data_df = gene_data_df[gene_data_df['id'].isin(target_genes)].copy()

In [ ]:
print(target_gene_data_df.shape)
target_gene_data_df.head()

In [ ]:
NEG_CTRL_TYPES = ['intergenic', 'non_targeting', 'splice_nonessential']
def assign_control(sgrna_type: str):
    if sgrna_type in NEG_CTRL_TYPES:
        return 'negative'
    elif sgrna_type == 'splice_essential':
        return 'positive'
    else:
        return None
good_data_df['control'] = good_data_df['sgRNA_type'].apply(assign_control)

In [ ]:
control_vc = good_data_df['control'].value_counts()
control_vc

In [ ]:
control_only_df = good_data_df[~good_data_df['control'].isna()].copy()
control_only_df = control_only_df.sort_values('LFC')

# Compute cutoffs

In [ ]:
def get_lfc_cutoffs(df, cell_lines, editors, fdr):
    lfc_cutoffs = defaultdict(dict) # d[cell_line][editor]
    for editor in editors:
        current_cell_lines = cell_lines
        for cell_line in current_cell_lines: 
            is_cl = df['cell_line'] == cell_line
            is_e = df['editor'] == editor
            cl_e_control_df = df[is_cl & is_e]
            
            cl_e_vc = cl_e_control_df['control'].value_counts()
            n_pos = cl_e_vc['positive']
            n_neg = cl_e_vc['negative']

            if fdr != None:
            
                lfc_values = cl_e_control_df['LFC'].values
                # LFC values were sorted earlier
                neg_fracs = []
                for lfc_i, lfc_value in enumerate(lfc_values):
                    subset_lfc_df = cl_e_control_df[cl_e_control_df['LFC'] <= lfc_value]
                    counts = subset_lfc_df.value_counts('control')
                    if not ('positive' in counts):
                        f_pos = 0
                    else:
                        f_pos = counts['positive'] / n_pos
                        
                    if not ('negative' in counts):
                        f_neg = 0
                    else:
                        f_neg = counts['negative'] / n_neg
                    
                    neg_frac = float(f_neg / (f_neg + f_pos))
        
                    neg_fracs.append(neg_frac)
        
                    if neg_frac <= fdr or (lfc_i == 0):
                    # if f_neg <= FDR or (lfc_i == 0):
                        last_f_neg = f_neg
                        last_f_pos = f_pos
                        last_lfc = lfc_value
                        lfc_cutoffs[cell_line][editor] = last_lfc

                # print(last_f_neg, last_f_pos)
            
            else:
                positive_lfcs = cl_e_control_df[cl_e_control_df['control'] == 'positive']['LFC'].values
                lfc_value = np.quantile(positive_lfcs, 0.95)
                lfc_cutoffs[cell_line][editor] = lfc_value

    return lfc_cutoffs

In [ ]:
lfc_cutoffs = get_lfc_cutoffs(control_only_df, cell_lines, editors, fdr=fdr)

In [ ]:
# LFC with FDR5% without refinement
lfc_cutoff_df = pd.DataFrame(lfc_cutoffs).reset_index(names='editor')
lfc_cutoff_df = lfc_cutoff_df.melt(id_vars='editor',
                                    value_vars=cell_lines, 
                                    var_name='cell_line', 
                                    value_name='lfc_cutoff')
lfc_cutoff_df = lfc_cutoff_df.sort_values(['editor', 'cell_line'])
lfc_cutoff_df

In [ ]:
palette_name = 'colorblind'
sns.set_palette(palette_name)
palette = sns.color_palette(palette_name)
blue, yellow, green, orange = palette[:4]

In [ ]:
df = control_only_df.copy()
df['Control'] = df['control'].replace({v: v.capitalize() for v in df['control'].unique()})
order = ['Negative', 'Positive']
f = sns.displot(data=df,
            x='LFC',
           hue='Control',
            row='editor',
            row_order=editors,
           col='cell_line',
            col_order=cell_lines,
           kind='kde',
           common_norm=False,
           hue_order=order,
           facet_kws=dict(sharex=False,
                         sharey=False),
               palette=[green, orange]
               )

# Add the cut-off lines to each subplot
for editor, editor_axes in zip(editors, f.axes):
    for cell_line, ax in zip(cell_lines, editor_axes):
        if editor in lfc_cutoffs[cell_line]:
            cutoff = lfc_cutoffs[cell_line][editor]
            ax.axvline(cutoff, c='grey', linestyle='--')

# After creating your displot and before saving it
from matplotlib.lines import Line2D

# Create a handle for the cut-off line
cutoff_handle = Line2D([0], [0], color='grey', linestyle='--', label='Cut-off')

# Get the existing legend
legend = f.figure.legends[0]  # Assuming there's only one legend in the figure

# Get existing handles and labels
handles = legend.get_lines()  # Using get_lines() instead of legendHandles
labels = [text.get_text() for text in legend.get_texts()]

# Add the new handle and label
handles.append(cutoff_handle)
labels.append('Cut-off')

# Remove the old legend and create a new one in the same position
legend_bbox = legend.get_bbox_to_anchor()
legend.remove()
f.figure.legend(handles, labels, 
                loc='right',
                # bbox_to_anchor=legend_bbox
                )


f.set_titles('{col_name} - {row_name}')
plt.savefig(os.path.join(figures_dirpath, 'ctrl_lfc_kde.png'))



In [ ]:
df.groupby(['editor', 'cell_line', 'Control'])['LFC'].median()

# Add DepMap data

In [ ]:
target_data_df = good_data_df[good_data_df['Gene'].isin(target_genes)].copy()

In [ ]:
target_data_df['logP'] = target_data_df['p.twosided'].apply(lambda x: -1*np.log10(x))

In [ ]:
depmap_gene_effects.head()

In [ ]:
depmap_gene_effects = depmap_gene_effects.rename(DEPMAP2CELL_LINE, axis=1)
depmap_gene_cell_line = depmap_gene_effects.melt(id_vars=['Gene'], 
                                                 value_vars=cell_lines, 
                                                 var_name='cell_line',
                                                value_name='gene_effect')

In [ ]:
depmap_gene_cell_line.head()

In [ ]:
dp_control_only_df = control_only_df.merge(depmap_gene_cell_line,
                                          on=['cell_line', 'Gene'])

In [ ]:
dp_control_only_df = dp_control_only_df.sort_values('gene_effect')

In [ ]:
ess_genes_dfs = []
for cell_line, cl_df in depmap_gene_cell_line.groupby('cell_line'):
    cl_df = cl_df[cl_df['Gene'].isin(target_genes)].copy()
    cl_df['is_essential'] = cl_df['gene_effect'] < gene_effect_essential_threshold
    ess_genes_dfs.append(cl_df)

ess_genes_df = pd.concat(ess_genes_dfs)

In [ ]:
cl2ess_genes = {cell_line: ess_genes_df[(ess_genes_df['cell_line'] == cell_line) & (ess_genes_df['is_essential'])]['Gene'].unique().tolist()
               for cell_line in cell_lines}

In [ ]:
for cell_line, genes in cl2ess_genes.items():
    print(cell_line)
    print(', '.join(genes))
    print(len(genes))
    print('')

In [ ]:
ess_genes = set(cl2ess_genes[cell_lines[0]])
for cell_line in cell_lines[1:]:
    ess_genes = ess_genes.intersection(cl2ess_genes[cell_line])

ess_genes_df['is_essential_all'] = ess_genes_df['Gene'].isin(ess_genes)

In [ ]:
control_only_df['control'].value_counts()

# Plot hit rates

In [ ]:
df = good_data_df.copy()
df = df.merge(lfc_cutoff_df,
                 on=['cell_line', 'editor'],
                 how='left')
df['lt_cutoff'] = df['LFC'] <= df['lfc_cutoff']

def get_category(row):
    control = row['control']
    # if control is not None:
    if pd.notna(control):
        sgrna_id = row['sgRNA_ID']
        editor = row['editor']
        return control.capitalize() + ' control'
    else:
        gene = row['Gene']
        cell_line = row['cell_line']
        ess_genes_cl = cl2ess_genes[cell_line]
        if gene in ess_genes_cl:
            return 'Essential target gene'
        else:
            return 'Non essential target gene'
df['Guide category'] = df.apply(get_category, axis=1)
df['Hit rate (%)'] = df['lt_cutoff'] * 100

In [ ]:
df['Guide category'].value_counts(dropna=False)

In [ ]:
order = ['Negative control', 
         'Non essential target gene', 
         'Essential target gene', 
         'Positive control']
f= sns.catplot(data=df,
               y='Guide category',
            x='Hit rate (%)',
           hue='Guide category',
            col='cell_line',
            col_order=cell_lines,
            row_order=editors,
            row='editor',
           kind='bar',
               order=order,
           hue_order=order,
               # hue_order=['negative', 'Non essential gene', 'Essential gene'],
            errorbar=None,
           sharex=False,
            # sharey=False,
              palette=[green, yellow, blue, orange]
              )
f.set_titles('{col_name} - {row_name}')
f.set_axis_labels(
    # x_var='Hit rate',
                 y_var='')
# for axes in f.axes:
#     for ax in axes:
#         ax.set_xticks(ax.get_xticks(), rotation=70)
# f.set_xticklabels(rotation=70)
plt.savefig(os.path.join(figures_dirpath, 'control_and_target_bar.png'), dpi=300)

In [ ]:
order = ['Negative control', 
         'Non essential target gene', 
         'Essential target gene']
f= sns.catplot(data=df,
               y='Guide category',
            x='Hit rate (%)',
           hue='Guide category',
            col='cell_line',
            col_order=cell_lines,
            row_order=editors,
            row='editor',
           kind='bar',
               order=order,
           hue_order=order,
               # hue_order=['negative', 'Non essential gene', 'Essential gene'],
            errorbar=None,
           sharex=False,
            # sharey=False,
              palette=[green, yellow, blue]
              )

for editor, editor_axes in zip(editors, f.axes):
    for cell_line, ax in zip(cell_lines, editor_axes):

        cl_e_df = df[(df['cell_line'] == cell_line) 
        & (df['editor'] == editor)
        & (df['Guide category'].isin(order))]

        mean_values = cl_e_df.groupby('Guide category')['Hit rate (%)'].mean()
        xposlist = [mean_values[e] + mean_values[e] * 0.05 for e in order]
        yposlist = range(len(xposlist))
        yposlist = [e for e in yposlist]

        
        counts = cl_e_df.groupby(['Guide category'])['Hit rate (%)'].count()
        # stringlist = [f'n =\n{counts[e]}' for e in order]
        stringlist = [f'n={counts[e]}' for e in order]
        # stringlist = ['n = 62','n = 19','n = 87','n = 76']
        
        for i in range(len(stringlist)):
            # print(xposlist[i], yposlist[i], stringlist[i])
            ax.text(xposlist[i], yposlist[i], stringlist[i], fontsize='small')

        ax.set_xlim(0, mean_values.max() * 1.5)
        # plt.ylim(-0.05, 1.1)


f.set_titles('{col_name} - {row_name}')
f.set_axis_labels(
    # x_var='Hit rate',
                 y_var='')
plt.savefig(os.path.join(figures_dirpath, 'control_and_target_bar_nopos.png'), dpi=300)

In [ ]:
df[~df['Guide category'].isin(control_order)].groupby(['editor', 'cell_line', 'Guide category']
                                               )['lt_cutoff'].mean().round(4).reset_index().pivot_table(
    index=['editor', 'cell_line'],
    columns=['Guide category'],
    values=['lt_cutoff'])

# More merging

In [ ]:
target_data_df = target_data_df.merge(depmap_gene_cell_line, 
                                on=['Gene', 'cell_line'], 
                                how='left')

In [ ]:
target_data_df = target_data_df.merge(ess_genes_df[['Gene', 'cell_line', 'is_essential']], 
                                      on=['cell_line', 'Gene'], 
                                      how='left')
target_data_df['is_essential'] = target_data_df['is_essential'].fillna(False)

In [ ]:
target_data_df = target_data_df.merge(lfc_cutoff_df, 
                                on=['cell_line', 'editor'], 
                                how='left')

In [ ]:
target_data_df[['Gene', 'is_essential']].value_counts()

In [ ]:
target_data_df['lt_cutoff'] = target_data_df['LFC'] <= target_data_df['lfc_cutoff']

In [ ]:
target_data_df.shape

In [ ]:
cutoffs = target_data_df.drop_duplicates(['lfc_cutoff'])[['cell_line', 'editor', 'lfc_cutoff', 'mean_lfc', 'std_lfc']]
cutoffs['zscore_cutoff'] = (cutoffs['lfc_cutoff'] - cutoffs['mean_lfc']) / cutoffs['std_lfc']

In [ ]:
cutoffs

# Disentangle single positions

In [ ]:
def get_single_positions(row):
    
    protein_positions = row['Protein_position']
    original_aas = []
    modified_aas = []
    amino_acids = row['Amino_acids']
    try:
        single_positions = [int(protein_positions)]
        if isinstance(amino_acids, str):
            aa_split = amino_acids.split('/')
            if len(aa_split) > 1:
                original_aa, modified_aa = aa_split
                if len(original_aa) != 1:
                    print('Something wrong')
                    print(amino_acids)
                    print(aa_split)
                original_aas.append(original_aa)
                modified_aas.append(modified_aa)
    except: # there's likely a dash in the Protein position
        pp_split = protein_positions.split('-')
        single_positions = ['-']
        if isinstance(amino_acids, str):
            assert len(pp_split) > 1
            aa_split = amino_acids.split('/')
            if len(aa_split) > 1:
            
                try:
                    protein_positions = list(range(int(pp_split[0]), 
                                                   int(pp_split[1]) + 1))
                    n_positions = len(protein_positions)
                    
                    modified_idxs = []
                    assert len(aa_split[0]) == len(protein_positions)
                    for i, (original_aa, modified_aa) in enumerate(zip(aa_split[0], aa_split[1])):
                        if original_aa != modified_aa:
                            modified_idxs.append(i)
                            original_aas.append(original_aa)
                            modified_aas.append(modified_aa)
                
                    single_positions = [protein_positions[idx] for idx in modified_idxs]
            
                except ValueError as ex:
                    print(ex)

    d_result = {'Single_positions' : single_positions,
               'Original_AA': original_aas,
               'Modified_AA': modified_aas}
    return pd.Series(d_result)

single_pos_df = multiple_vep_df.apply(get_single_positions, axis=1)

In [ ]:
multiple_vep_df.shape

In [ ]:
single_pos_df.head()

In [ ]:
multiple_vep_df = pd.concat([multiple_vep_df, single_pos_df], axis=1)

In [ ]:
multiple_vep_df = multiple_vep_df.explode(column=list(single_pos_df.columns))

In [ ]:
# Initially, the multiple mutation have a slice of protein_position
# This is not optimal because if we have a mutation in 4 and 9
# The middle protein_position is not mutated
# This part is to have have one row per protein_position altered by guide
multiple_vep_df['Protein_position'] = multiple_vep_df['Single_positions']
multiple_vep_df = multiple_vep_df.drop('Single_positions', axis=1)

In [ ]:
multiple_vep_df[['Protein_position', 'Original_AA', 'Modified_AA']].head()

In [ ]:
single_pos_df = single_vep_df.apply(get_single_positions, axis=1)

In [ ]:
single_vep_df = pd.concat([single_vep_df, single_pos_df], axis=1)

In [ ]:
print(single_vep_df.shape)
single_vep_df = single_vep_df.explode(column=list(single_pos_df.columns))
single_vep_df = single_vep_df.drop('Single_positions', axis=1)
print(single_vep_df.shape)

In [ ]:
multiple_vep_df['Edit_Location'] = multiple_vep_df['Edit_Location'].astype(int)
multiple_vep_df['Protein_position'] = multiple_vep_df['Protein_position'].astype(str)
single_vep_df['Protein_position'] = single_vep_df['Protein_position'].astype(str)

In [ ]:
single_combos = single_vep_df['Feature'] + '_' + single_vep_df['Edit_Location'].astype(str) + '_' + single_vep_df['Protein_position']
multiple_combos = multiple_vep_df['Feature'] + '_' + multiple_vep_df['Edit_Location'].astype(str) + '_' + multiple_vep_df['Protein_position']

In [ ]:
# Single combo defines the real combinations between the edit location and the protein position
single_combos = single_vep_df[['Feature', 'Edit_Location', 'Protein_position']].copy()
single_combos = single_combos.drop_duplicates()
single_combos['in_single_combo'] = True

In [ ]:
# This removes the wrong edit_location for single amino_acid mutations
# (since multiple_vep_df is cross-joining its edit_locations with each variant annotation)
# e.g. Edit Location 104780136 gets associated with Protein_position 2 while
# it should only count for the Protein_position 1

print(multiple_vep_df.shape)
multiple_vep_df = multiple_vep_df.merge(single_combos,
                     on=['Feature', 'Edit_Location', 'Protein_position'],
                     how='left')
print(multiple_vep_df.shape)
multiple_vep_df = multiple_vep_df[((multiple_vep_df['Consequence'].str.contains('missense')) 
& (multiple_vep_df['in_single_combo']))
| (~multiple_vep_df['Consequence'].str.contains('missense'))]
print(multiple_vep_df.shape)

In [ ]:
single_and_multiple = pd.concat([single_vep_df, multiple_vep_df]).reset_index()

# Even more merging

In [ ]:
selected_columns = ['sgRNA_ID', 'CRISPR_PAM_Sequence', 'Location', 'Edit_Location', 
                    'Direction', 'Strand', 'editor', '#Uploaded_variation', 'Consequence',
                   'Protein_position', 'Amino_acids', 'Codons', 'Multiple_Location',
                   'Original_AA', 'Modified_AA']

In [ ]:
with_mut_df = target_data_df.merge(single_and_multiple[selected_columns], 
                            left_on=['sgRNA_ID', 'editor'], 
                            right_on=['sgRNA_ID', 'editor'],
                           how='left')
with_mut_df.shape

# Process consequences

In [ ]:
def remove_comma(s):
    if isinstance(s, str) and ',' in s:
        return s.split(',')[0]
    else:
        return s
with_mut_df['Consequence'] = with_mut_df['Consequence'].apply(remove_comma)

In [ ]:
def remove_suffix(s):
    if isinstance(s, str):
        return s.removesuffix('_variant')
    else:
        return s

with_mut_df['Consequence'] = with_mut_df['Consequence'].apply(remove_suffix)

In [ ]:
# Unknown are guides that lead to no mutation (e.g. no A in the 4-9 window for a ABE experiment)
with_mut_df['Consequence'] = with_mut_df['Consequence'].fillna('unknown')

In [ ]:
# high_imp_splice = ['splice_acceptor', 'splice_donor']
# low_imp_splice = ['splice_donor_5th_base', 'splice_region', 
#                   'splice_donor_region', 'splice_polypyrimidine_tract']

# consequence_order = ['stop_gained', 'stop_lost', 'start_lost'] \
#                     + [cons for cons in with_mut_df['Consequence'].unique() 
#                        if isinstance(cons, str) and cons in high_imp_splice] \
#                     + ['missense']\
#                     + [cons for cons in with_mut_df['Consequence'].unique() 
#                        if isinstance(cons, str) and cons in low_imp_splice] \
#                     + ['synonymous', 'stop_retained', 'intron'] \
#                     + [cons for cons in with_mut_df['Consequence'].unique() 
#                        if isinstance(cons, str) and 'UTR' in cons and not ',' in cons] \
#                     + ['unknown', 'wt', 'coding_sequence', ]

selected_splice = ['splice_acceptor', 'splice_donor', 'splice_donor_region']

consequence_order = ['stop_gained', 'stop_lost', 'start_lost'] \
                    + [cons for cons in with_mut_df['Consequence'].unique() 
                       if isinstance(cons, str) and cons in selected_splice] \
                    + ['missense']\
                    + ['synonymous', 'stop_retained', 'intron'] \
                    + [cons for cons in with_mut_df['Consequence'].unique() 
                       if isinstance(cons, str) and 'UTR' in cons and not ',' in cons] \
                    + ['unknown', 'wt', 'coding_sequence', ]

In [ ]:
with_mut_df['Consequence'] = pd.Categorical(with_mut_df['Consequence'], consequence_order)
# with_mut_df = with_mut_df.sort_values(['cell_line', 'editor', 'sgrna', 'Consequence', 'Protein_position'])

# Add BLOSUM data

In [ ]:
BLOSUM = bl.BLOSUM(62)

In [ ]:
def get_blosum(aa_str):
    try:
        aa1, aa2 = aa_str.split('/')
        for aa in [aa1, aa2]:
            assert aa != '*'
            assert len(aa) >= 1
        assert len(aa1) == len(aa2)
        if len(aa1) > 1:
            values = []
            for a1, a2 in zip(aa1, aa2):
                if a1 != a2:
                    values.append(BLOSUM[a1][a2])
            if len(values) > 0:
                value = np.min(values)
            else:
                value = np.nan # multiple mutation but synonymous
        else:
            value = BLOSUM[aa1][aa2]
    except:
        value = np.nan
    return value

In [ ]:
with_mut_df['blosum'] = with_mut_df['Amino_acids'].apply(get_blosum)

In [ ]:
with_mut_df['blosum'].value_counts(dropna=False)

In [ ]:
# MIN_BLOSUM_CONSV = -1
# def get_conservative(blosum: float):
#     if np.isnan(blosum):
#         value = np.nan
#     elif blosum >= MIN_BLOSUM_CONSV:
#         value = True
#     else:
#         value = False
#     return value
    
# with_mut_df['is_conservative'] = with_mut_df['blosum'].apply(get_conservative)

In [ ]:
AA_TYPES = {
    'aliphatic': 'GAVLI',
    'polar': 'SCUTM',
    'cyclic': 'P',
    'aromatic': 'FYW',
    'basic': 'HKR',
    'acidic-amide': 'DENQ'
}

AA2TYPE = {aa: aatype  
           for aatype, aas in AA_TYPES.items()
           for aa in aas}

def get_conservative(aa_str):
    try:
        aas1, aas2 = aa_str.split('/')
        for aas in [aas1, aas2]:
            assert aas != '*'
            assert len(aas) >= 1
        assert len(aas1) == len(aas2)
        
        values = []
        for aa1, aa2 in zip(aas1, aas2):
            if aa1 != aa2:
                aa_type1 = AA2TYPE[aa1]
                aa_type2 = AA2TYPE[aa2]
                value = aa_type1 == aa_type2
                values.append(value)
        if len(values) > 0:
            value = all(values)
        else:
            value = np.nan # multiple mutation but synonymous
            
    except:
        value = np.nan
        
    return value

with_mut_df['is_conservative'] = with_mut_df['Amino_acids'].apply(get_conservative)

In [ ]:
with_mut_df['is_conservative'].value_counts()

In [ ]:
with_mut_df = with_mut_df.sort_values(['cell_line', 'editor', 'sgRNA_ID', 'Consequence', 'is_conservative', 'Protein_position'])

# Add Variant classes (model)

In [ ]:
variant_pred_df = variant_pred_df.rename({'Variant class ': 'Variant class'}, axis=1)
variant_pred_df['Protein_position'] = variant_pred_df['uniprot_rno'].astype(float)

In [ ]:
with_mut_df = with_mut_df.merge(variant_pred_df[['Gene', 'Protein_position', 'Original_AA', 'Modified_AA', 'Variant class']], 
                            on=['Gene', 'Protein_position', 'Original_AA', 'Modified_AA'],
                           how='left')

In [ ]:
with_mut_df.head(2)

In [ ]:
variant_order = ['Total-loss', 'SBI', 'WT-like']
with_mut_df['Variant class'] = pd.Categorical(with_mut_df['Variant class'], variant_order)

In [ ]:
with_mut_df['Variant class'].value_counts()

# Add editing window

In [ ]:
def get_ew_position(row):
    direction = row['Direction']
    bool_direction = direction == 'right'
    strand = row['Strand']
    bool_strand = strand == 1
    edit_location = row['Edit_Location']
    location = row['Location']
    if isinstance(location, str):
        locations = row['Location'].split(':')[1]
        start_location, end_location = locations.split('-')
        start_location = int(start_location)
        end_location = int(end_location)
        locations = list(range(start_location, end_location + 1))
    
        # right and -1, or left and 1 : opposite strands between CRISPR and gene
        if not bool_direction:
            locations = list(reversed(locations))

        return locations.index(edit_location) + 1 # + 1 to go back to 1-indexed position
    else:
        return np.nan

In [ ]:
with_mut_df['ew_position'] = with_mut_df.apply(get_ew_position, axis=1)

In [ ]:
with_mut_df['ew_position'].value_counts()

# Add off targets (+ plots)

In [ ]:
with_mut_df_w_ot = with_mut_df.copy()
with_mut_df = with_mut_df.merge(ot_df[['sgRNA_ID', 'is_ot', 'cell_line']],
                               on=['sgRNA_ID', 'cell_line'])

In [ ]:
with_mut_df['Gene'].unique()

In [ ]:
pd.DataFrame(with_mut_df[with_mut_df['is_essential']].drop_duplicates('sgRNA_ID')['Gene'].value_counts())

In [ ]:
sns.catplot(data=with_mut_df,
           y='lt_cutoff',
            kind='bar',
            errorbar=None,
            x='is_ot',
            row='editor',
            row_order=editors,
            col='cell_line',
            col_order=cell_lines,
            # common_norm=False,
           )

# Save main dataframe

In [ ]:
exported_columns = ['Gene', 'sgRNA_ID', 'LFC', 'p.twosided', 'logP', 'cell_line', 'editor', 'zscore', 
                    'CRISPR_PAM_Sequence', 'Location', 'Edit_Location', 'Multiple_Location', 'Direction',
                   'Consequence', 'Protein_position', 'Amino_acids',
                    # 'HGVSp',
                    'gene_effect', 'lfc_cutoff', 'lt_cutoff', 'blosum', 'is_conservative',
                   'Original_AA', 'Modified_AA', 'Variant class', 'is_essential',
                   'ew_position', 'Strand', 'Codons', 'is_ot']

In [ ]:
for gene in target_genes:
    gene_dirpath = os.path.join(processed_files_dirpath, gene)
    os.makedirs(gene_dirpath, exist_ok=True)
    gene_df = with_mut_df[with_mut_df['Gene'] == gene][exported_columns]
    gene_df.to_csv(os.path.join(gene_dirpath, f'{gene}_all_results.csv'), index=False)

In [ ]:
# Removing duplicate guide, keeping the worst consequence
with_mut_df = with_mut_df.sort_values(['cell_line', 'editor', 'sgRNA_ID', 'Consequence', 'is_conservative', 'Variant class', 'Protein_position'])
no_dup_guide_df = with_mut_df.drop_duplicates(['cell_line', 'editor', 'sgRNA_ID']).copy()

In [ ]:
no_dup_guide_df['Consequence'].value_counts()

In [ ]:
# Sorting first by edit location; for instance, a location can have a 
# splice_donor consequence, and a missense, but only the missense will have 
# a protein location. If LFC is low, it is probably due to the splice_donor
# mutation, not by a potentially conservative missense.
with_mut_df = with_mut_df.sort_values(['cell_line', 'editor', 'Gene', 'Edit_Location', 'Consequence', 'is_conservative', 'Protein_position', 'Variant class', 'LFC'])
no_dup_location_df = with_mut_df.drop_duplicates(['cell_line', 'editor', 'Gene', 'Edit_Location', 'Protein_position']).copy()
no_dup_location_df = no_dup_location_df.dropna(subset=['Edit_Location'])

In [ ]:
no_dup_location_df['Consequence'].value_counts()

In [ ]:
no_dup_location_df = no_dup_location_df.sort_values(['cell_line', 'editor', 'Gene', 'Protein_position', 'Consequence', 'is_conservative', 'Variant class', 'LFC'])
no_dup_position_df = no_dup_location_df.drop_duplicates(['cell_line', 'editor', 'Gene', 'Protein_position']).copy()
no_dup_position_df = no_dup_position_df[~no_dup_position_df['Protein_position'].isin(['-', 'nan'])]

In [ ]:
# We rank by Consequence before LFC because we might have guides that 
# have low LFC but only does synonymous mutation to a protein position
# of interest. We'd rather keep a LFC -0.5 for a missense than a LFC -2
# for a synonymous, the LFC -2 is due to other residues modified by the guide
# with_mut_df = with_mut_df.sort_values(['cell_line', 'editor', 'Gene', 'Protein_position', 'Consequence', 'is_conservative', 'Variant class', 'LFC'])
# no_dup_position_df = with_mut_df.drop_duplicates(['cell_line', 'editor', 'Gene', 'Protein_position']).copy()
# no_dup_position_df = no_dup_position_df[~no_dup_position_df['Protein_position'].isin(['-', 'nan'])]

In [ ]:
no_dup_position_df['Consequence'].value_counts(dropna=False)

In [ ]:
no_dup_position_df.shape

In [ ]:
no_dup_guide_df.shape

In [ ]:
no_dup_guide_df['is_conservative'].value_counts(dropna=False)

# Compute main_conseq

In [ ]:
def get_main_conseq(conseq):
    if conseq == 'stop_gained':
        return 'stop_gained'
    elif conseq == 'start_lost':
        return 'start_lost'
    # elif conseq in high_imp_splice:
    #     return 'high_imp_splice'
    elif conseq in selected_splice:
        return 'splice'
    elif conseq == 'missense':
        return 'missense'
    # elif conseq in low_imp_splice:
    #     return 'low_imp_splice'
    elif conseq == 'synonymous':
        return 'synonymous'
    elif conseq == 'intron':
        return 'intron'
    elif conseq == 'unknown':
        return 'unknown'
    else:
        return 'other'

In [ ]:
no_dup_guide_df['main_conseq'] = no_dup_guide_df['Consequence'].apply(get_main_conseq)
no_dup_position_df['main_conseq'] = no_dup_position_df['Consequence'].apply(get_main_conseq)

In [ ]:
no_dup_guide_df['main_conseq'].value_counts()

In [ ]:
no_dup_position_df['main_conseq'].value_counts(dropna=False)

In [ ]:
d_consq_consv_replace = {
    'synonymous_nan': 'synonymous',
    'missense_False': 'missense_non_conservative',
    'missense_True': 'missense_conservative',
    'splice_nan': 'splice',
    'stop_gained_nan': 'stop_gained',
    'start_lost_False': 'start_lost',
    'start_lost_True': 'start_lost',
    'intron_nan': 'intron'
}

no_dup_guide_df['consq_consv'] = no_dup_guide_df['main_conseq'] + '_' + no_dup_guide_df['is_conservative'].astype(str)
no_dup_guide_df['consq_consv'] = no_dup_guide_df['consq_consv'].replace(d_consq_consv_replace)

no_dup_position_df['consq_consv'] = no_dup_position_df['main_conseq'] + '_' + no_dup_position_df['is_conservative'].astype(str)
no_dup_position_df['consq_consv'] = no_dup_position_df['consq_consv'].replace(d_consq_consv_replace)

In [ ]:
no_dup_guide_df['consq_consv'].value_counts()

In [ ]:
no_dup_position_df['consq_consv'].value_counts()

# Save no duplicates dataframes

In [ ]:
exported_columns = ['sgRNA_ID', 'LFC', 'p.twosided', 'cell_line', 'editor', 'zscore', 
                    'CRISPR_PAM_Sequence', 'Location', 'Edit_Location', 'Direction',
                   'Consequence', 'Protein_position', 'Amino_acids',
                   'logP', 'Gene', 'gene_effect', 'lfc_cutoff', 'lt_cutoff', 'blosum', 
                   'is_conservative', 'main_conseq', 'consq_consv', 'Original_AA', 
                    'Modified_AA', 'is_essential', 'Variant class', 'ew_position',
                   'Strand', 'Codons','is_ot']

In [ ]:
for gene in target_genes:
    gene_dirpath = os.path.join(processed_files_dirpath, gene)
    os.makedirs(gene_dirpath, exist_ok=True)
    gene_df = no_dup_position_df[no_dup_position_df['Gene'] == gene][exported_columns]
    gene_df.to_csv(os.path.join(gene_dirpath, f'{gene}_results_no_dup_position.csv'), index=False)
    
    gene_df = no_dup_guide_df[no_dup_guide_df['Gene'] == gene][exported_columns]
    gene_df.to_csv(os.path.join(gene_dirpath, f'{gene}_results_no_dup_guide.csv'), index=False)

In [ ]:
exported_columns = ['sgRNA_ID', 'LFC', 'p.twosided', 'cell_line', 'editor', 'zscore', 
                    'CRISPR_PAM_Sequence', 'Location', 'Edit_Location', 'Direction',
                   'Consequence', 'Protein_position', 'Amino_acids',
                   'logP', 'Gene', 'gene_effect', 'lfc_cutoff', 'lt_cutoff', 'blosum', 
                   'is_conservative', 'Original_AA', 
                    'Modified_AA', 'is_essential', 'Variant class', 'ew_position',
                   'Strand', 'Codons', 'is_ot']

for gene in target_genes:
    gene_dirpath = os.path.join(processed_files_dirpath, gene)
    os.makedirs(gene_dirpath, exist_ok=True)
    gene_df = no_dup_location_df[no_dup_location_df['Gene'] == gene][exported_columns]
    gene_df.to_csv(os.path.join(gene_dirpath, f'{gene}_results_no_dup_location.csv'), index=False)

# Plots

In [ ]:
df = no_dup_guide_df.copy()
frac_df = df.groupby(['cell_line', 'editor', 'Gene', 'is_essential'])[['lt_cutoff', 'gene_effect']].mean().reset_index()
frac_df['fraction_hit_guides'] = frac_df['lt_cutoff']
df = frac_df
df = df[df['editor'] == 'ABE']
df = df.sort_values(['is_essential', 'fraction_hit_guides', 'cell_line'])
plt.figure(figsize=(16, 8))
f = sns.barplot(data=df,
                y='fraction_hit_guides',
               x='Gene',
                hue='cell_line',
                hue_order=cell_lines,
               errorbar=None)
# f.set_title(f'{cell_line} - {editor}')
f.set_ylabel('Hit rate')
plt.xticks(rotation=70)
plt.show()